# Chapter 12: Special Methods for Sequences
*From: Fluent Python by Luciano Ramalho*

---

This chapter builds a multidimensional **Vector** class, step by step, to demonstrate
the special methods that make a user-defined type behave like a real Python sequence:
`__len__`, `__getitem__`, `__getattr__`, `__hash__`, `__eq__`, and `__format__`.

Along the way we explore **protocols and duck typing** -- the idea that conforming to
an informal interface (implementing the right methods) is all you need.

## TL;DR

- Store components in an **`array.array`** of floats; accept any iterable in the constructor.
- Implement `__len__` + `__getitem__` to satisfy the **sequence protocol** (duck typing).
- Handle **`slice`** objects in `__getitem__` so slicing returns a new `Vector`, not a raw `array`.
- Use `__getattr__` for shorthand like `v.x`, `v.y` -- but **pair it with `__setattr__`** to prevent silent inconsistencies.
- Hash with `functools.reduce(operator.xor, ...)` over all components (map-reduce pattern).
- Efficient `__eq__`: compare lengths first, then use `zip` + `all` for short-circuit comparison.
- Extend the **format mini-language** with a custom `'h'` suffix for hyperspherical coordinates.

---
## 1. Vector: A Multidimensional Sequence (Baseline)

Our `Vector` stores float components in an `array.array('d', ...)` and accepts
any iterable in its constructor. We use `reprlib.repr` to keep the representation
safe for vectors with thousands of dimensions.

In [ ]:
from array import array
import reprlib
import math

class Vector:
    typecode = 'd'

    def __init__(self, components):
        self._components = array(self.typecode, components)

    def __iter__(self):
        return iter(self._components)

    def __repr__(self):
        # reprlib.repr limits the output length (adds '...')
        components = reprlib.repr(self._components)
        # Strip 'array(\'d\', ' prefix and trailing ')'
        components = components[components.find('['):-1]
        return f'Vector({components})'

    def __str__(self):
        return str(tuple(self))

    def __eq__(self, other):
        return tuple(self) == tuple(other)

    def __abs__(self):
        return math.hypot(*self)

    def __bool__(self):
        return bool(abs(self))

    def __bytes__(self):
        return bytes([ord(self.typecode)]) + bytes(self._components)

    @classmethod
    def frombytes(cls, octets):
        typecode = chr(octets[0])
        memv = memoryview(octets[1:]).cast(typecode)
        return cls(memv)

# --- Quick demo ---
print(Vector([3.1, 4.2]))
print(Vector((3, 4, 5)))
print(Vector(range(10)))        # reprlib truncates long output
print(f'abs = {abs(Vector([3, 4]))}')


---
## 2. Protocols and Duck Typing

A **protocol** in Python is an *informal interface* defined by convention, not by code.
If your class implements `__len__` and `__getitem__`, it **is** a sequence -- regardless
of its class hierarchy. This is **duck typing**: "If it quacks like a duck..."

> You don't check whether it *is* a duck: you check whether it *quacks* like a duck.
> -- Alex Martelli

In [ ]:
import collections

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck:
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit)
                       for suit in self.suits
                       for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

# FrenchDeck inherits from object, yet it IS a sequence
deck = FrenchDeck()
print(f'len = {len(deck)}')
print(f'First: {deck[0]}')
print(f'Last:  {deck[-1]}')
print(f'Slice: {deck[12::13]}')  # the four aces

# Iteration works without __iter__ -- Python falls back to __getitem__
from random import choice, seed
seed(42)
print(f'Random card: {choice(deck)}')
print(f'King of hearts in deck? {Card("K", "hearts") in deck}')

**Key insight:** You can even get away with implementing *part* of a protocol.
For example, `__getitem__` alone is enough to support iteration (Python calls
`__getitem__` with indices 0, 1, 2, ... until `IndexError`). You only need
`__len__` if something actually calls `len()` on your object.

Since Python 3.8, `typing.Protocol` offers **static protocols** (checked by type
checkers like mypy), while the traditional ones are **dynamic protocols** enforced
only at runtime.

---
## 3. A Sliceable Sequence: `__getitem__` with Slice Support

When you write `v[1:4]`, Python passes a `slice(1, 4, None)` object to
`__getitem__`. We detect this and return a new `Vector` from the sliced array.
For integer keys, we use `operator.index()` instead of `int()` -- because
`int(3.14)` silently returns 3, while `operator.index(3.14)` correctly raises
`TypeError` (floats should never be indices).

In [ ]:
# Let's see what Python passes to __getitem__

class MySeq:
    def __getitem__(self, index):
        return index

s = MySeq()
print(f's[1]     = {s[1]!r}')
print(f's[1:4]   = {s[1:4]!r}')
print(f's[1:4:2] = {s[1:4:2]!r}')
print(f's[1:4, 9] = {s[1:4, 9]!r}')  # tuple with a slice and an int

In [ ]:
import operator
from array import array
import reprlib
import math

class Vector:
    typecode = 'd'

    def __init__(self, components):
        self._components = array(self.typecode, components)

    def __repr__(self):
        components = reprlib.repr(self._components)
        components = components[components.find('['):-1]
        return f'Vector({components})'

    def __len__(self):
        return len(self._components)

    def __getitem__(self, key):
        if isinstance(key, slice):
            cls = type(self)
            return cls(self._components[key])  # return a new Vector
        index = operator.index(key)            # raises TypeError for floats
        return self._components[index]

    def __iter__(self):
        return iter(self._components)

    def __abs__(self):
        return math.hypot(*self)

    def __bool__(self):
        return bool(abs(self))

# --- Demo ---
v7 = Vector(range(7))
print(f'v7 = {v7}')
print(f'v7[-1]  = {v7[-1]}')        # single item -> float
print(f'v7[1:4] = {v7[1:4]}')       # slice -> new Vector!
print(f'v7[-1:] = {v7[-1:]}')       # single-element slice -> Vector
print(f'len(v7) = {len(v7)}')

# This correctly raises TypeError (tuple cannot be an index):
try:
    v7[1, 2]
except TypeError as e:
    print(f'v7[1, 2] -> TypeError: {e}')

---
## 4. Dynamic Attribute Access: `__getattr__` and `__setattr__`

We want `v.x`, `v.y`, `v.z`, `v.t` as shortcuts for `v[0]`...`v[3]`.
`__getattr__` is called as a **fallback** when normal attribute lookup fails.

**The trap:** If you only implement `__getattr__`, then `v.x = 10` silently
creates an instance attribute, and subsequent reads of `v.x` find that attribute
*before* `__getattr__` is ever called. The underlying component stays unchanged --
a silent inconsistency. The fix: implement `__setattr__` to block writes.

In [ ]:
import operator
from array import array
import reprlib
import math

class Vector:
    typecode = 'd'
    __match_args__ = ('x', 'y', 'z', 't')  # also enables positional pattern matching

    def __init__(self, components):
        self._components = array(self.typecode, components)

    def __repr__(self):
        components = reprlib.repr(self._components)
        components = components[components.find('['):-1]
        return f'Vector({components})'

    def __len__(self):
        return len(self._components)

    def __getitem__(self, key):
        if isinstance(key, slice):
            cls = type(self)
            return cls(self._components[key])
        index = operator.index(key)
        return self._components[index]

    def __getattr__(self, name):
        cls = type(self)
        try:
            pos = cls.__match_args__.index(name)
        except ValueError:
            pos = -1
        if 0 <= pos < len(self._components):
            return self._components[pos]
        msg = f'{cls.__name__!r} object has no attribute {name!r}'
        raise AttributeError(msg)

    def __setattr__(self, name, value):
        cls = type(self)
        if len(name) == 1:
            if name in cls.__match_args__:
                error = 'readonly attribute {attr_name!r}'
            elif name.islower():
                error = "can't set attributes 'a' to 'z' in {cls_name!r}"
            else:
                error = ''
            if error:
                msg = error.format(cls_name=cls.__name__, attr_name=name)
                raise AttributeError(msg)
        super().__setattr__(name, value)

    def __iter__(self):
        return iter(self._components)

    def __abs__(self):
        return math.hypot(*self)

    def __bool__(self):
        return bool(abs(self))

# --- Demo ---
v = Vector(range(10))
print(f'v.x = {v.x}')       # 0.0 -- reads v[0]
print(f'v.y = {v.y}')       # 1.0
print(f'v.z = {v.z}')       # 2.0
print(f'v.t = {v.t}')       # 3.0

# __setattr__ blocks writes to single lowercase letters:
try:
    v.x = 10
except AttributeError as e:
    print(f'v.x = 10 -> AttributeError: {e}')

try:
    v.a = 99
except AttributeError as e:
    print(f"v.a = 99 -> AttributeError: {e}")

# But multi-letter attributes are fine (normal behavior):
v.name = 'my_vector'
print(f'v.name = {v.name!r}')

---
## 5. Hashing and a Faster `==`: `__hash__` with reduce, `__eq__` with zip

To make `Vector` hashable, we XOR the hashes of all components using
`functools.reduce(operator.xor, ...)` -- a classic **map-reduce** pattern:

1. **Map**: compute `hash(x)` for each component
2. **Reduce**: combine them with `^` (xor) into a single integer

For `__eq__`, converting both operands to tuples is wasteful for large vectors.
Instead, we compare lengths first, then use `zip` + `all` for short-circuit comparison.

In [ ]:
import functools
import operator

# --- Understanding reduce ---
# reduce(fn, iterable) applies fn to pairs cumulatively: fn(fn(fn(a,b),c),d)

# Example: factorial with reduce
print('5! =', functools.reduce(lambda a, b: a * b, range(1, 6)))

# Aggregate XOR three ways:
# 1) for loop
n = 0
for i in range(1, 6):
    n ^= i
print(f'XOR via loop: {n}')

# 2) reduce with lambda
print(f'XOR via lambda: {functools.reduce(lambda a, b: a ^ b, range(6))}')

# 3) reduce with operator.xor (preferred!)
print(f'XOR via operator.xor: {functools.reduce(operator.xor, range(6))}')

In [ ]:
import functools
import operator
from array import array
import reprlib
import math

class Vector:
    typecode = 'd'
    __match_args__ = ('x', 'y', 'z', 't')

    def __init__(self, components):
        self._components = array(self.typecode, components)

    def __repr__(self):
        components = reprlib.repr(self._components)
        components = components[components.find('['):-1]
        return f'Vector({components})'

    def __len__(self):
        return len(self._components)

    def __getitem__(self, key):
        if isinstance(key, slice):
            cls = type(self)
            return cls(self._components[key])
        index = operator.index(key)
        return self._components[index]

    def __iter__(self):
        return iter(self._components)

    def __abs__(self):
        return math.hypot(*self)

    def __bool__(self):
        return bool(abs(self))

    # --- Efficient __eq__: length check + zip + all ---
    def __eq__(self, other):
        return (len(self) == len(other) and
                all(a == b for a, b in zip(self, other)))

    # --- __hash__: map-reduce with xor ---
    def __hash__(self):
        hashes = (hash(x) for x in self)             # map step
        return functools.reduce(operator.xor, hashes, 0)  # reduce step

# --- Demo ---
v1 = Vector([3, 4])
v2 = Vector([3, 4])
v3 = Vector([3, 4, 5])
v6 = Vector(range(6))

print(f'v1 == v2: {v1 == v2}')       # True
print(f'v1 == v3: {v1 == v3}')       # False (different lengths)

print(f'hash(v1) = {hash(v1)}')
print(f'hash(v3) = {hash(v3)}')
print(f'hash(v6) = {hash(v6)}')

# Hashable -> can be used as dict key or in sets
vectors = {v1: '2D', v3: '3D'}
print(f'dict lookup: vectors[Vector([3,4])] = {vectors[Vector([3, 4])]!r}')
print(f'set: {len({v1, v2, v3})} unique vectors')

**Why `reduce` with an initializer?**
Always pass the third argument to `reduce`: `reduce(fn, iterable, initializer)`.
Without it, `reduce` on an empty iterable raises `TypeError`.
For XOR, the identity value is `0` (since `x ^ 0 == x`).

---
## 6. Custom Formatting: `__format__` with Hyperspherical Coordinates

We extend the format mini-language with a `'h'` suffix for **hyperspherical
coordinates**: `<r, phi1, phi2, ...>` where `r` is the magnitude and
the remaining values are angular coordinates.

Without `'h'`, the vector formats in Cartesian: `(x, y, z, ...)`.

We use `itertools.chain` to seamlessly iterate over magnitude + angles,
and generator expressions throughout for memory efficiency.

In [ ]:
import itertools
import functools
import operator
from array import array
import reprlib
import math

class Vector:
    typecode = 'd'
    __match_args__ = ('x', 'y', 'z', 't')

    def __init__(self, components):
        self._components = array(self.typecode, components)

    def __repr__(self):
        components = reprlib.repr(self._components)
        components = components[components.find('['):-1]
        return f'Vector({components})'

    def __len__(self):
        return len(self._components)

    def __getitem__(self, key):
        if isinstance(key, slice):
            cls = type(self)
            return cls(self._components[key])
        index = operator.index(key)
        return self._components[index]

    def __iter__(self):
        return iter(self._components)

    def __abs__(self):
        return math.hypot(*self)

    def __bool__(self):
        return bool(abs(self))

    def __eq__(self, other):
        return (len(self) == len(other) and
                all(a == b for a, b in zip(self, other)))

    def __hash__(self):
        hashes = (hash(x) for x in self)
        return functools.reduce(operator.xor, hashes, 0)

    def __getattr__(self, name):
        cls = type(self)
        try:
            pos = cls.__match_args__.index(name)
        except ValueError:
            pos = -1
        if 0 <= pos < len(self._components):
            return self._components[pos]
        msg = f'{cls.__name__!r} object has no attribute {name!r}'
        raise AttributeError(msg)

    def __setattr__(self, name, value):
        cls = type(self)
        if len(name) == 1:
            if name in cls.__match_args__:
                error = 'readonly attribute {attr_name!r}'
            elif name.islower():
                error = "can't set attributes 'a' to 'z' in {cls_name!r}"
            else:
                error = ''
            if error:
                msg = error.format(cls_name=cls.__name__, attr_name=name)
                raise AttributeError(msg)
        super().__setattr__(name, value)

    # --- Hyperspherical coordinate support ---

    def angle(self, n):
        """Compute the n-th angular coordinate (1-indexed)."""
        r = math.hypot(*self[n:])
        a = math.atan2(r, self[n - 1])
        if (n == len(self) - 1) and (self[-1] < 0):
            return math.pi * 2 - a
        return a

    def angles(self):
        """Generator yielding all angular coordinates."""
        return (self.angle(n) for n in range(1, len(self)))

    def __format__(self, fmt_spec=''):
        if fmt_spec.endswith('h'):               # hyperspherical
            fmt_spec = fmt_spec[:-1]
            coords = itertools.chain([abs(self)], self.angles())
            outer_fmt = '<{}>'
        else:                                     # Cartesian (default)
            coords = self
            outer_fmt = '({})'
        components = (format(c, fmt_spec) for c in coords)
        return outer_fmt.format(', '.join(components))

    def __str__(self):
        return str(tuple(self))

    def __bytes__(self):
        return bytes([ord(self.typecode)]) + bytes(self._components)

    @classmethod
    def frombytes(cls, octets):
        typecode = chr(octets[0])
        memv = memoryview(octets[1:]).cast(typecode)
        return cls(memv)


# --- Demo: Cartesian formatting ---
v1 = Vector([3, 4])
print(f'Default:  {format(v1)}')
print(f'.2f:      {format(v1, ".2f")}')
print(f'.3e:      {format(v1, ".3e")}')

v3 = Vector([3, 4, 5])
print(f'3D default: {format(v3)}')

# --- Demo: Hyperspherical formatting ---
print(f'2D spherical:  {format(Vector([1, 1]), "h")}')
print(f'3D .3e:        {format(Vector([2, 2, 2]), ".3eh")}')
print(f'4D .5f:        {format(Vector([0, 1, 0, 0]), "0.5fh")}')
print(f'4D negative:   {format(Vector([-1, -1, -1, -1]), "h")}')

---
## Try It Yourself

### Exercise 1: Implement `__contains__` for membership testing

Our `Vector` already supports `in` via `__getitem__` fallback iteration, but
it does a sequential scan. Implement `__contains__` that does the same but
explicitly, so you understand the protocol.

In [ ]:
# Exercise 1: Add __contains__ to a simple sequence class
class NumberLine:
    def __init__(self, values):
        self._values = array('d', values)

    def __len__(self):
        return len(self._values)

    def __getitem__(self, key):
        return self._values[key]

    def __contains__(self, value):
        """Explicit membership test."""
        return any(v == value for v in self._values)

    def __repr__(self):
        return f'NumberLine({list(self._values)})'

nl = NumberLine([1.0, 2.0, 3.0, 4.0, 5.0])
print(f'3.0 in nl: {3.0 in nl}')
print(f'9.0 in nl: {9.0 in nl}')
print(f'len(nl):   {len(nl)}')
print(f'nl[2]:     {nl[2]}')

### Exercise 2: Build a hashable `Point3D`

Create a `Point3D(x, y, z)` class that:
- Supports `__getitem__` (indexing by 0, 1, 2)
- Is hashable (`__hash__` via XOR, `__eq__` via zip)
- Has a `__format__` that shows Cartesian by default or spherical with `'s'`

In [ ]:
# Exercise 2: Your solution
import functools
import operator
import math

class Point3D:
    __slots__ = ('_coords',)

    def __init__(self, x, y, z):
        object.__setattr__(self, '_coords', (x, y, z))

    def __getitem__(self, index):
        return self._coords[index]

    def __len__(self):
        return 3

    def __iter__(self):
        return iter(self._coords)

    def __repr__(self):
        return f'Point3D({self._coords[0]!r}, {self._coords[1]!r}, {self._coords[2]!r})'

    def __eq__(self, other):
        return len(self) == len(other) and all(a == b for a, b in zip(self, other))

    def __hash__(self):
        return functools.reduce(operator.xor, (hash(c) for c in self), 0)

    def __format__(self, fmt_spec=''):
        if fmt_spec.endswith('s'):  # spherical
            fmt_spec = fmt_spec[:-1]
            r = math.sqrt(sum(c ** 2 for c in self))
            theta = math.acos(self[2] / r) if r != 0 else 0.0
            phi = math.atan2(self[1], self[0])
            coords = (format(c, fmt_spec) for c in (r, theta, phi))
            return '<' + ', '.join(coords) + '>'
        else:
            coords = (format(c, fmt_spec) for c in self)
            return '(' + ', '.join(coords) + ')'

# Test it
p = Point3D(1.0, 1.0, 1.0)
print(f'repr:     {p!r}')
print(f'hash:     {hash(p)}')
print(f'p == p:   {p == Point3D(1.0, 1.0, 1.0)}')
print(f'Cartesian:  {format(p, ".2f")}')
print(f'Spherical:  {format(p, ".4fs")}')

# Use in a set
points = {Point3D(1, 0, 0), Point3D(0, 1, 0), Point3D(1, 0, 0)}
print(f'Unique points: {len(points)}')

### Exercise 3: `__getattr__` pitfall

Create a class with `__getattr__` but *without* `__setattr__`. Demonstrate the
inconsistency bug, then fix it. This solidifies the lesson from Section 4.

In [ ]:
# Exercise 3: Demonstrate the __getattr__ pitfall

class BuggyVec:
    """Has __getattr__ but NO __setattr__ guard -- buggy!"""

    def __init__(self, x, y):
        self._data = [x, y]

    def __getattr__(self, name):
        if name == 'x':
            return self._data[0]
        if name == 'y':
            return self._data[1]
        raise AttributeError(f'no attribute {name!r}')

    def __repr__(self):
        return f'BuggyVec({self._data[0]}, {self._data[1]})'

bv = BuggyVec(1.0, 2.0)
print(f'bv.x = {bv.x}')        # 1.0 (via __getattr__)
bv.x = 99                       # creates instance attribute!
print(f'bv.x = {bv.x}')        # 99 (reads instance attr, __getattr__ never called)
print(f'bv._data = {bv._data}') # [1.0, 2.0] -- underlying data unchanged!
print('BUG: bv.x and bv._data[0] disagree!')
print()

# --- Fixed version ---
class FixedVec:
    def __init__(self, x, y):
        self._data = [x, y]

    def __getattr__(self, name):
        if name == 'x':
            return self._data[0]
        if name == 'y':
            return self._data[1]
        raise AttributeError(f'no attribute {name!r}')

    def __setattr__(self, name, value):
        if name in ('x', 'y'):
            raise AttributeError(f'readonly attribute {name!r}')
        super().__setattr__(name, value)

    def __repr__(self):
        return f'FixedVec({self._data[0]}, {self._data[1]})'

fv = FixedVec(1.0, 2.0)
print(f'fv.x = {fv.x}')
try:
    fv.x = 99
except AttributeError as e:
    print(f'fv.x = 99 -> {e}')
print('FIXED: writes are blocked, no inconsistency.')

---
## Key Takeaways

1. **Sequence protocol = `__len__` + `__getitem__`**. That is all Python needs to treat your object as a sequence (iteration, `in`, slicing, `sorted`, `reversed`).

2. **Duck typing**: What matters is behavior, not class hierarchy. Implement the right methods and your class *is* the type.

3. **Slicing returns `slice` objects**. Detect them in `__getitem__` with `isinstance(key, slice)` and return a new instance of your class, not a raw array.

4. **`operator.index()` over `int()`** for index conversion. It rejects floats, which should never be used as indices.

5. **`__getattr__` needs `__setattr__`**. Without it, attribute assignment creates instance attributes that shadow the dynamic lookup, causing silent bugs.

6. **Map-reduce hashing**: `functools.reduce(operator.xor, (hash(x) for x in self), 0)` -- efficient, lazy, and correct for empty sequences.

7. **`zip` + `all` for `__eq__`**: Compare lengths first (zip stops silently at the shortest), then short-circuit on first mismatch.

8. **Extend the format mini-language** by detecting a custom suffix in `__format__` and choosing between coordinate systems.

---
## See Also

- [[vector-multidimensional-sequence]] -- Building the baseline Vector class
- [[protocols-and-duck-typing]] -- Informal interfaces and structural subtyping
- [[sliceable-sequence-getitem]] -- `__getitem__` with slice support
- [[dynamic-attribute-access]] -- `__getattr__` and `__setattr__`
- [[hashing-and-eq]] -- `__hash__` with reduce, `__eq__` with zip
- [[formatting-with-format]] -- Custom `__format__` with hyperspherical coordinates

**Next:** Chapter 13 covers Interfaces, Protocols, and ABCs in depth.